In [0]:
%run ../utils/init.py

In [0]:
files = dbutils.fs.ls(f"{RAW}/insee/populations/")
for f in files:
    print(f.name, f.size)

In [0]:
df = spark.read.parquet(f"{RAW}/insee/populations/populations_2021.parquet")
df.printSchema()
df.show(2)

In [0]:
from pyspark.sql.functions import current_timestamp, lpad

IDF_DEPTS = ['75', '77', '78', '91', '92', '93', '94', '95']

df_bronze = (spark.read.parquet(f"{RAW}/insee/populations/populations_2021.parquet")
    # Cast code_departement double → string propre ("1.0" → "01")
    .withColumn("code_departement", 
        lpad(F.col("code_departement").cast("int").cast("string"), 2, "0"))
    # Filtre IDF
    .filter(F.col("code_departement").isin(IDF_DEPTS))
    .withColumn("annee_recensement", F.lit(2021))
    .withColumn("ingestion_timestamp", current_timestamp())
    .drop("population_comptee_a_part")
)

print(f"Lignes IDF : {df_bronze.count()}")
df_bronze.show(5)

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("code_departement") \
    .save(f"{BRONZE}/insee_populations/")

print("✅ Bronze INSEE Populations écrit")